# Exotic Rates

**Purpose:** Demonstrate the deterministic coupon and payoff helpers behind several exotic rate structures.

**Prerequisites:** Familiarity with coupon mechanics, path-dependent products, and CMS-style rate references.

**In this notebook:** We inspect coupon profiles for a TARN, a snowball, and an inverse floater, then compute a CMS spread option intrinsic payoff and a callable range-accrual coupon.


## Concept

These helpers focus on deterministic payoff logic rather than full Monte Carlo pricing. That makes them especially helpful for:

1. Understanding how coupon state evolves path by path.
2. Sanity-checking intrinsic payoff formulas.
3. Building intuition before using the full pricing pipeline.


In [ ]:
import numpy as np

from finstack.valuations import (
    callable_range_accrual_accrued,
    cms_spread_option_intrinsic,
    snowball_coupon_profile,
    tarn_coupon_profile,
)

rng = np.random.default_rng(seed=42)


## Coupon profiles and intrinsic payoffs

The next cell reuses the script inputs directly so you can inspect how each product responds to a simple path or fixing set. Notebook format works well here because each printed block is effectively a worked example.


In [ ]:
tarn = tarn_coupon_profile(
    fixed_rate=0.08,
    coupon_floor=0.0,
    floating_fixings=np.linspace(0.06, 0.015, 10).tolist(),
    target_coupon=0.20,
    day_count_fraction=0.5,
)
print("TARN coupons (%)      :", [round(100 * x, 3) for x in tarn["coupons_paid"]])
print("TARN cumulative (%)   :", [round(100 * x, 3) for x in tarn["cumulative"]])
print("TARN redemption index :", tarn["redemption_index"])

snowball = snowball_coupon_profile(
    initial_coupon=0.05,
    fixed_rate=0.06,
    floating_fixings=np.linspace(0.05, 0.01, 8).tolist(),
    floor=0.0,
    cap=float("inf"),
    is_inverse_floater=False,
    leverage=1.0,
)
print("\nSnowball coupons (%) :", [round(100 * x, 3) for x in snowball])

inverse = snowball_coupon_profile(
    initial_coupon=0.0,
    fixed_rate=0.08,
    floating_fixings=np.linspace(0.01, 0.06, 8).tolist(),
    floor=0.0,
    cap=float("inf"),
    is_inverse_floater=True,
    leverage=2.0,
)
print("Inverse floater (%)   :", [round(100 * x, 3) for x in inverse])

cms_payoff = cms_spread_option_intrinsic(
    long_cms=0.045,
    short_cms=0.033,
    strike=0.005,
    is_call=True,
    notional=10_000_000.0,
)
print(f"\nCMS spread option intrinsic: ${cms_payoff:,.2f}")

observations = rng.normal(loc=0.035, scale=0.01, size=250).tolist()
range_accrual = callable_range_accrual_accrued(
    lower=0.02,
    upper=0.05,
    observations=observations,
    coupon_rate=0.05,
    day_count_fraction=250.0 / 360.0,
)
in_range = sum(1 for obs in observations if 0.02 <= obs <= 0.05)
print(f"Range-accrual in-range observations: {in_range} / {len(observations)}")
print(f"Range-accrual coupon (% notional) : {100 * range_accrual:.3f}%")


## Takeaways

- The deterministic helpers are ideal for understanding product mechanics before you bring in model or market assumptions.
- `tarn_coupon_profile()` and `snowball_coupon_profile()` make path dependence explicit.
- `cms_spread_option_intrinsic()` and `callable_range_accrual_accrued()` are convenient sanity checks for structured-rate payoff logic.


In [ ]:
{
    "tarn_redeemed_early": tarn["redeemed_early"],
    "snowball_last_coupon": round(snowball[-1], 6),
    "inverse_last_coupon": round(inverse[-1], 6),
    "cms_intrinsic": round(cms_payoff, 2),
    "range_accrual": round(range_accrual, 6),
}
